In [ ]:
import sys
from pathlib import Path

# Add project root to Python path
project_root = Path("..").resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print("Project root added to path:", project_root)

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
# from sklearn.metrics import mean_squared_error

# Project imports
from src.preprocessing import get_preprocessor
from src.crossvalidation import get_cv
from src.models import get_rf_pipeline

In [ ]:
DATA_PATH = Path("../data/raw/TotalDataSet.xlsx")
df = pd.read_excel(DATA_PATH)

def assign_perf_group(gpa):
    if gpa <= 2.99:
        return "Underperforming"
    elif gpa <= 3.29:
        return "Average"
    else:
        return "Performing"

df["GroupLabel"] = df["FinalGPA"].apply(assign_perf_group)

In [ ]:
numeric_features = [
    'Zscore'
]
numeric_features_sem1 = [
    'Zscore','S1',
]
numeric_features_sem2 = [
    'Zscore', 'S1', 'S2',
]
numeric_features_sem3 = [
    'Zscore', 'S1', 'S2', 'S3',
]
numeric_features_sem4 = [
    'Zscore', 'S1', 'S2', 'S3', 'S4',
]
numeric_features_sem5 = [
    'Zscore', 'S1', 'S2', 'S3', 'S4', 'S5'
]
numeric_features_sem6 = [
    'Zscore', 'S1', 'S2', 'S3', 'S4', 'S5', 'S6',
]
numeric_features_sem7 = [
    'Zscore', 'S1', 'S2', 'S3', 'S4', 'S5', 'S6', 'S7',
]
numeric_features_sem8 = [
    'Zscore',
    'S1', 'S2', 'S3', 'S4', 'S5', 'S6', 'S7', 'S8'
]
categorical_features = [
    'Gender', 'Department', 'District', 'MediumAL'
]

y = df["FinalGPA"]
strata = df["GroupLabel"]

Semester 0

In [ ]:
import sklearn.metrics

X = df[numeric_features + categorical_features]

preprocessor = get_preprocessor(
    numeric_features=numeric_features,
    categorical_features=categorical_features
)

rf_model = get_rf_pipeline(
    preprocessor=preprocessor,
    max_depth=3,
    n_estimators=100,
)

cv = get_cv()
groups = df["GroupLabel"].unique()

# Containers for results
rf_results = {g: {'rmse': [], 'r2': [], 'bias': []} for g in groups}
overall_rmse_list = []
overall_r2_list = []

for train_idx, val_idx in cv.split(X, strata):
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
    g_val = strata.iloc[val_idx]

    # Fit and Predict
    rf_model.fit(X_train, y_train)
    y_pred = rf_model.predict(X_val)

    rmse_sem0 = sklearn.metrics.root_mean_squared_error(y_val, y_pred)
    r2_sem0 = sklearn.metrics.r2_score(y_val, y_pred)
    
    overall_rmse_list.append(rmse_sem0)
    overall_r2_list.append(r2_sem0)

    # Calculate metrics per group
    for g in groups:
        mask = g_val == g
        if mask.sum() > 0:
            actual = y_val[mask]
            pred = y_pred[mask]
            
            rf_results[g]['rmse'].append(sklearn.metrics.root_mean_squared_error(actual, pred))
            rf_results[g]['r2'].append(sklearn.metrics.r2_score(actual, pred))
            rf_results[g]['bias'].append(np.mean(actual - pred))

In [ ]:
print("\n--- OVERALL PERFORMANCE (All Students) ---")
print(f"Mean RMSE: {np.mean(overall_rmse_list):.3f} (SD: {np.std(overall_rmse_list):.3f})")
print(f"Mean R2:   {np.mean(overall_r2_list):.3f} (SD: {np.std(overall_r2_list):.3f})")

# 2. PRINT GROUP-WISE PERFORMANCE
print("\n--- GROUP-WISE PERFORMANCE ---")
for g in groups:
    # Convert lists to numpy arrays for easier math
    g_rmse_s0 = np.array(rf_results[g]['rmse'])
    g_r2_s0 = np.array(rf_results[g]['r2'])
    g_bias_s0 = np.array(rf_results[g]['bias'])
    
    print(f"\n>> Group: {g}")
    print(f"   RMSE: {np.mean(g_rmse_s0):.3f} (SD: {np.std(g_rmse_s0):.3f})")
    print(f"   R2:   {np.mean(g_r2_s0):.3f} (SD: {np.std(g_r2_s0):.3f})")
    print(f"   Bias: {np.mean(g_bias_s0):.3f} (SD: {np.std(g_bias_s0):.3f})")


SEMESTER 1

In [ ]:
preprocessor_s1 = get_preprocessor(
    numeric_features=numeric_features_sem1,
    categorical_features=categorical_features
)

rf_model_1 = get_rf_pipeline(
    preprocessor=preprocessor_s1,
    max_depth=3,
    n_estimators=100,
)

X = df[numeric_features_sem1 + categorical_features]

cv = get_cv()
groups = df["GroupLabel"].unique()

# Containers for results
rf_results_1 = {g: {'rmse': [], 'r2': [], 'bias': []} for g in groups}
overall_rmse_1 = []
overall_r2_1 = []

for train_idx, val_idx in cv.split(X, strata):
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
    g_val = strata.iloc[val_idx]

    # Fit and Predict
    rf_model_1.fit(X_train, y_train)
    y_pred_1 = rf_model_1.predict(X_val)

    rmse_sem1 = sklearn.metrics.root_mean_squared_error(y_val, y_pred_1)
    r2_sem1 = sklearn.metrics.r2_score(y_val, y_pred_1)

    overall_rmse_1.append(rmse_sem1)
    overall_r2_1.append(r2_sem1)

    # Calculate metrics per group
    for g in groups:
        mask = g_val == g
        if mask.sum() > 0:
            actual = y_val[mask]
            pred = y_pred_1[mask]

            rf_results_1[g]['rmse'].append(sklearn.metrics.root_mean_squared_error(actual, pred))
            rf_results_1[g]['r2'].append(sklearn.metrics.r2_score(actual, pred))
            rf_results_1[g]['bias'].append(np.mean(actual - pred))

In [ ]:
print("\n--- OVERALL PERFORMANCE (All Students) ---")
print(f"Mean RMSE: {np.mean(overall_rmse_1):.3f} (SD: {np.std(overall_rmse_1):.3f})")
print(f"Mean R2:   {np.mean(overall_r2_1):.3f} (SD: {np.std(overall_r2_1):.3f})")

# 2. PRINT GROUP-WISE PERFORMANCE
print("\n--- GROUP-WISE PERFORMANCE ---")
for g in groups:
    # Convert lists to numpy arrays for easier math
    g_rmse_s1 = np.array(rf_results_1[g]['rmse'])
    g_r2_s1 = np.array(rf_results_1[g]['r2'])
    g_bias_s1 = np.array(rf_results_1[g]['bias'])
    
    print(f"\n>> Group: {g}")
    print(f"   RMSE: {np.mean(g_rmse_s1):.3f} (SD: {np.std(g_rmse_s1):.3f})")
    print(f"   R2:   {np.mean(g_r2_s1):.3f} (SD: {np.std(g_r2_s1):.3f})")
    print(f"   Bias: {np.mean(g_bias_s1):.3f} (SD: {np.std(g_bias_s1):.3f})")

SEMESTER 2

In [ ]:
preprocessor_s2 = get_preprocessor(
    numeric_features=numeric_features_sem2,
    categorical_features=categorical_features
)

rf_model_2 = get_rf_pipeline(
    preprocessor=preprocessor_s2,
    max_depth=5,
    n_estimators=100,
)

X = df[numeric_features_sem2 + categorical_features]

cv = get_cv()
groups = df["GroupLabel"].unique()

# Containers for results
rf_results_2 = {g: {'rmse': [], 'r2': [], 'bias': []} for g in groups}
overall_rmse_2 = []
overall_r2_2 = []

for train_idx, val_idx in cv.split(X, strata):
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
    g_val = strata.iloc[val_idx]

    # Fit and Predict
    rf_model_2.fit(X_train, y_train)
    y_pred_2 = rf_model_2.predict(X_val)

    rmse_sem2 = sklearn.metrics.root_mean_squared_error(y_val, y_pred_2)
    r2_sem2 = sklearn.metrics.r2_score(y_val, y_pred_2)

    overall_rmse_2.append(rmse_sem2)
    overall_r2_2.append(r2_sem2)

    # Calculate metrics per group
    for g in groups:
        mask = g_val == g
        if mask.sum() > 0:
            actual = y_val[mask]
            pred = y_pred_2[mask]

            rf_results_2[g]['rmse'].append(sklearn.metrics.root_mean_squared_error(actual, pred))
            rf_results_2[g]['r2'].append(sklearn.metrics.r2_score(actual, pred))
            rf_results_2[g]['bias'].append(np.mean(actual - pred))

In [ ]:
print("\n--- OVERALL PERFORMANCE (All Students) ---")
print(f"Mean RMSE: {np.mean(overall_rmse_2):.3f} (SD: {np.std(overall_rmse_2):.3f})")
print(f"Mean R2:   {np.mean(overall_r2_2):.3f} (SD: {np.std(overall_r2_2):.3f})")

# 2. PRINT GROUP-WISE PERFORMANCE
print("\n--- GROUP-WISE PERFORMANCE ---")
for g in groups:
    # Convert lists to numpy arrays for easier math
    g_rmse_s2 = np.array(rf_results_2[g]['rmse'])
    g_r2_s2 = np.array(rf_results_2[g]['r2'])
    g_bias_s2 = np.array(rf_results_2[g]['bias'])

    print(f"\n>> Group: {g}")
    print(f"   RMSE: {np.mean(g_rmse_s2):.3f} (SD: {np.std(g_rmse_s2):.3f})")
    print(f"   R2:   {np.mean(g_r2_s2):.3f} (SD: {np.std(g_r2_s2):.3f})")
    print(f"   Bias: {np.mean(g_bias_s2):.3f} (SD: {np.std(g_bias_s2):.3f})")

SEMESTER 3

In [ ]:
preprocessor_s3 = get_preprocessor(
    numeric_features=numeric_features_sem3,
    categorical_features=categorical_features
)

rf_model_3 = get_rf_pipeline(
    preprocessor=preprocessor_s3,
    max_depth=3,
    n_estimators=100,
)

X = df[numeric_features_sem3 + categorical_features]

cv = get_cv()
groups = df["GroupLabel"].unique()

# Containers for results
rf_results_3 = {g: {'rmse': [], 'r2': [], 'bias': []} for g in groups}
overall_rmse_3 = []
overall_r2_3 = []

for train_idx, val_idx in cv.split(X, strata):
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
    g_val = strata.iloc[val_idx]

    # Fit and Predict
    rf_model_3.fit(X_train, y_train)
    y_pred_3 = rf_model_3.predict(X_val)

    rmse_sem3 = sklearn.metrics.root_mean_squared_error(y_val, y_pred_3)
    r2_sem3 = sklearn.metrics.r2_score(y_val, y_pred_3)

    overall_rmse_3.append(rmse_sem3)
    overall_r2_3.append(r2_sem3)

    # Calculate metrics per group
    for g in groups:
        mask = g_val == g
        if mask.sum() > 0:
            actual = y_val[mask]
            pred = y_pred_3[mask]

            rf_results_3[g]['rmse'].append(sklearn.metrics.root_mean_squared_error(actual, pred))
            rf_results_3[g]['r2'].append(sklearn.metrics.r2_score(actual, pred))
            rf_results_3[g]['bias'].append(np.mean(actual - pred))

In [ ]:
print("\n--- OVERALL PERFORMANCE (SEM3) ---")
print(f"Mean RMSE: {np.mean(overall_rmse_3):.3f} (SD: {np.std(overall_rmse_3):.3f})")
print(f"Mean R2:   {np.mean(overall_r2_3):.3f} (SD: {np.std(overall_r2_3):.3f})")

# 2. PRINT GROUP-WISE PERFORMANCE
print("\n--- GROUP-WISE PERFORMANCE ---")
for g in groups:
    # Convert lists to numpy arrays for easier math
    g_rmse_s3 = np.array(rf_results_3[g]['rmse'])
    g_r2_s3 = np.array(rf_results_3[g]['r2'])
    g_bias_s3 = np.array(rf_results_3[g]['bias'])

    print(f"\n>> Group: {g}")
    print(f"   RMSE: {np.mean(g_rmse_s3):.3f} (SD: {np.std(g_rmse_s3):.3f})")
    print(f"   R2:   {np.mean(g_r2_s3):.3f} (SD: {np.std(g_r2_s3):.3f})")
    print(f"   Bias: {np.mean(g_bias_s3):.3f} (SD: {np.std(g_bias_s3):.3f})")

preprocessor = get_preprocessor(
    numeric_features=numeric_features,
    categorical_features=categorical_features
)

rf_model = get_rf_pipeline(
    preprocessor=preprocessor,
    n_estimators=100,
)

cv = get_cv()
groups = df["GroupLabel"].unique()

# Containers for results
rf_results = {g: {'rmse': [], 'r2': [], 'bias': []} for g in groups}
overall_rmse_list = []
overall_r2_list = []

for train_idx, val_idx in cv.split(X, strata):
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
    g_val = strata.iloc[val_idx]

    # Fit and Predict
    rf_model.fit(X_train, y_train)
    y_pred = rf_model.predict(X_val)

    rmse_sem0 = sklearn.metrics.root_mean_squared_error(y_val, y_pred)
    r2_sem0 = sklearn.metrics.r2_score(y_val, y_pred)
    
    overall_rmse_list.append(rmse_sem0)
    overall_r2_list.append(r2_sem0)

    # Calculate metrics per group
    for g in groups:
        mask = g_val == g
        if mask.sum() > 0:
            actual = y_val[mask]
            pred = y_pred[mask]
            
            rf_results[g]['rmse'].append(sklearn.metrics.root_mean_squared_error(actual, pred))
            rf_results[g]['r2'].append(sklearn.metrics.r2_score(actual, pred))
            rf_results[g]['bias'].append(np.mean(actual - pred))

In [ ]:
preprocessor_s1 = get_preprocessor(
    numeric_features=numeric_features_sem1,
    categorical_features=categorical_features
)

rf_model_1 = get_rf_pipeline(
    preprocessor=preprocessor_s1,
    n_estimators=100,
)

X = df[numeric_features_sem1 + categorical_features]

cv = get_cv()
groups = df["GroupLabel"].unique()

# Containers for results
rf_results_1 = {g: {'rmse': [], 'r2': [], 'bias': []} for g in groups}
overall_rmse_1 = []
overall_r2_1 = []

for train_idx, val_idx in cv.split(X, strata):
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
    g_val = strata.iloc[val_idx]

    # Fit and Predict
    rf_model_1.fit(X_train, y_train)
    y_pred_1 = rf_model_1.predict(X_val)

    rmse_sem1 = sklearn.metrics.root_mean_squared_error(y_val, y_pred_1)
    r2_sem1 = sklearn.metrics.r2_score(y_val, y_pred_1)

    overall_rmse_1.append(rmse_sem1)
    overall_r2_1.append(r2_sem1)

    # Calculate metrics per group
    for g in groups:
        mask = g_val == g
        if mask.sum() > 0:
            actual = y_val[mask]
            pred = y_pred_1[mask]

            rf_results_1[g]['rmse'].append(sklearn.metrics.root_mean_squared_error(actual, pred))
            rf_results_1[g]['r2'].append(sklearn.metrics.r2_score(actual, pred))
            rf_results_1[g]['bias'].append(np.mean(actual - pred))

We conducted a comparative baseline analysis between Ridge Regression and Random Forest. Despite the non-linear capabilities of Random Forest, Ridge Regression provided superior stability ($R^2$) and lower error (RMSE) in the critical early semesters (Sem 1-3). Given the constraints of the dataset size and the priority of early intervention, Ridge was selected as the primary model for the remainder of the longitudinal study